# 承認済み部位別 before / after を解析する

`video_region_pair_candidates.ipynb` で固定した `selected_region_pairs.json` を使います。
現在選択されている部位だけを解析し、別部位のペアへ自動で置き換えません。


In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_selected_regions.py')
if (cwd / marker).is_file():
    REPO_ROOT = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    REPO_ROOT = cwd.parent
    os.chdir(REPO_ROOT)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('repo root:', Path.cwd())


In [ ]:
VIDEO = Path('makeup0923.mp4')
OUTPUT = Path('outputs') / VIDEO.stem
SELECTED = OUTPUT / 'selected_region_pairs.json'
ANALYSIS_ROOT = OUTPUT / 'selected_region_analysis'

if not SELECTED.is_file():
    raise FileNotFoundError(f'部位別ペアが固定されていません: {SELECTED}')
print('selected:', SELECTED)


In [ ]:
from analysis.analyze_selected_regions import ANALYSIS_VERSION, analyze_selected_regions
EXPECTED_ANALYSIS_VERSION = 'selected-region-appearance-v10'
if ANALYSIS_VERSION != EXPECTED_ANALYSIS_VERSION:
    raise RuntimeError(
        f'解析モジュールが古いです: loaded={ANALYSIS_VERSION}, expected={EXPECTED_ANALYSIS_VERSION}. '
        'Jupyter kernel を再起動して Run All してください。'
    )
from IPython.display import HTML, display
import base64

summary = analyze_selected_regions(SELECTED, ANALYSIS_ROOT)
RUN_DIR = Path(summary['output_dir'])
print('analysis:', RUN_DIR)

# report.html 内の相対画像パスは VS Code/Jupyter の HTML 表示では
# notebook 側を基準に解決されて壊れるため、表示時だけ data URI に埋め込む。
report_html = (RUN_DIR / 'report.html').read_text(encoding='utf-8')
for region in summary['regions']:
    for phase in ('before', 'after'):
        filename = f'{region}_{phase}.png'
        image_path = RUN_DIR / filename
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        encoded = base64.b64encode(image_path.read_bytes()).decode('ascii')
        report_html = report_html.replace(
            f"src='{filename}'",
            f"src='data:image/png;base64,{encoded}'",
        )
display(HTML(report_html))


In [ ]:
for region, result in summary['regions'].items():
    print('\n', region)
    for row in result['deltas']:
        before = row['before']
        after = row['after']
        delta = row['delta']
        print(f"{row['label']}: {before:.4f} -> {after:.4f}  delta={delta:+.4f}")
print('\nreport:', RUN_DIR / 'report.html')
print('csv   :', RUN_DIR / 'feature_deltas.csv')


## 複数の before / after 候補で再現性を確認する

固定済み rank 1 は変更せず、保存済みの品質判定を使って候補を3秒間隔で再生成し、上位20ペアを同じ指標で横断比較します。候補母集団は最大80件まで生成します。顔サイズ・手重なりの閾値は変更しません。候補を承認済みペアへ昇格する処理ではありません。


In [ ]:
from analysis.analyze_region_rank_set import RANK_SET_VERSION, analyze_region_rank_set
EXPECTED_RANK_SET_VERSION = 'candidate-rank-set-v2'
if RANK_SET_VERSION != EXPECTED_RANK_SET_VERSION:
    raise RuntimeError(
        f'複数rank解析モジュールが古いです: loaded={RANK_SET_VERSION}, expected={EXPECTED_RANK_SET_VERSION}. '
        'Jupyter kernel を再起動して Run All してください。'
    )

REVIEW_COUNT = 20
REVIEW_RANKS = tuple(range(1, REVIEW_COUNT + 1))
CANDIDATE_TOP_K = 80
DIVERSITY_SECONDS = 3.0
RANK_SET_ROOT = OUTPUT / 'eye_texture_rank_validation'
rank_summary = analyze_region_rank_set(
    SELECTED,
    region='eye_texture',
    ranks=REVIEW_RANKS,
    output_root=RANK_SET_ROOT,
    candidate_top_k=CANDIDATE_TOP_K,
    diversity_seconds=DIVERSITY_SECONDS,
)
rank_run_dir = Path(rank_summary['output_dir'])

rank_report_html = (rank_run_dir / 'report.html').read_text(encoding='utf-8')
for pair in rank_summary['pairs']:
    for phase in ('before', 'after'):
        filename = pair['images'][phase]
        image_path = rank_run_dir / filename
        if not image_path.is_file():
            raise FileNotFoundError(image_path)
        encoded = base64.b64encode(image_path.read_bytes()).decode('ascii')
        rank_report_html = rank_report_html.replace(
            f"src='{filename}'",
            f"src='data:image/png;base64,{encoded}'",
        )
display(HTML(rank_report_html))
print('multi-rank report:', rank_run_dir / 'report.html')
print('multi-rank csv   :', rank_run_dir / 'feature_deltas.csv')


## 20ペアの傾向を集計する

同一人物・同一動画の候補20ペアについて、各指標の `after - before` の符号がどれだけそろうかを確認します。これは独立した20人のサンプルではなく、この動画内での再現性確認です。


In [ ]:
import pandas as pd

rows = []
for pair in rank_summary['pairs']:
    selection = pair['selection']
    gate = selection.get('region_gate', {})
    terms = selection.get('terms', {})
    deltas = {row['id']: row['delta'] for row in pair['deltas']}
    rows.append({
        'rank': pair['rank'],
        'score': selection['score'],
        'before_t': selection['before']['timestamp_seconds'],
        'after_t': selection['after']['timestamp_seconds'],
        'gap_sec': selection['after']['timestamp_seconds'] - selection['before']['timestamp_seconds'],
        'face_scale_ratio': selection['face_scale_ratio'],
        'before_hand_overlap': gate.get('before_hand_overlap_ratio'),
        'after_hand_overlap': gate.get('after_hand_overlap_ratio'),
        **{f'term__{key}': value for key, value in terms.items()},
        **{f'delta__{key}': value for key, value in deltas.items()},
    })

pair_df = pd.DataFrame(rows).sort_values('rank').reset_index(drop=True)
if len(pair_df) != REVIEW_COUNT:
    raise RuntimeError(f'期待した {REVIEW_COUNT} ペアではありません: actual={len(pair_df)}')

display(pair_df)
print('term columns:')
for column in [c for c in pair_df.columns if c.startswith('term__')]:
    print(' ', column)


In [ ]:
FOCUS_IDS = (
    'screen_left_upper_lid_skin_highpass_median_pct',
    'screen_left_upper_lid_skin_highpass_p90_pct',
    'screen_right_upper_lid_skin_highpass_median_pct',
    'screen_right_upper_lid_skin_highpass_p90_pct',
    'screen_left_nasolabial_crease_darkness_p90_pct',
    'screen_right_nasolabial_crease_darkness_p90_pct',
)

summary_rows = []
for metric_id in FOCUS_IDS:
    column = f'delta__{metric_id}'
    if column not in pair_df.columns:
        raise KeyError(f'必要な指標列がありません: {column}')
    values = pair_df[column].dropna()
    if values.empty:
        raise ValueError(f'有効値がありません: {column}')

    positive = int((values > 0).sum())
    negative = int((values < 0).sum())
    zero = int((values == 0).sum())
    if positive >= negative:
        majority_direction = '+'
        majority_count = positive
    else:
        majority_direction = '-'
        majority_count = negative

    summary_rows.append({
        'metric': metric_id,
        'n': int(len(values)),
        'positive': positive,
        'negative': negative,
        'zero': zero,
        'majority_direction': majority_direction,
        'consistency': majority_count / len(values),
        'median_delta': float(values.median()),
        'mean_delta': float(values.mean()),
        'std_delta': float(values.std(ddof=1)) if len(values) >= 2 else None,
        'min_delta': float(values.min()),
        'max_delta': float(values.max()),
    })

sign_summary = pd.DataFrame(summary_rows).sort_values(
    ['consistency', 'metric'], ascending=[False, True]
).reset_index(drop=True)
display(sign_summary)


In [ ]:
import matplotlib.pyplot as plt

for metric_id in FOCUS_IDS:
    column = f'delta__{metric_id}'
    plt.figure(figsize=(8, 4))
    plt.plot(pair_df['rank'], pair_df[column], marker='o')
    plt.axhline(0)
    plt.xlabel('candidate rank')
    plt.ylabel('after - before')
    plt.title(metric_id)
    plt.show()


## 上頬のプレーン肌 ROI で荒れ感を確認する

眉下やほうれい線そのものではなく、ほうれい線帯の外側にある比較的プレーンな上頬の肌を exploratory ROI として測ります。各 rank について ROI のアップを表示し、その直下に high-pass 中央値・high-pass p90・GVR-inspired の before / after / 差分を並べます。


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

from analysis.analyze_selected_regions import _load_phase
from analysis.appearance_features import (
    TEXTURE_BLUR_SIGMA,
    GVR_INSPIRED_CLAHE_CLIP_LIMIT,
    GVR_INSPIRED_CLAHE_TILE_GRID,
)

if 'rank_summary' not in globals():
    raise RuntimeError('rank_summary がありません。先に複数rank解析セルを実行してください。')


def build_upper_cheek_texture_mask(image, masks, side):
    candidate_name = f'{side}_nasolabial_candidate'
    control_name = f'{side}_nasolabial_outer_control'
    for name in (candidate_name, control_name):
        if name not in masks:
            raise KeyError(f'必要なROIがありません: {name}')

    candidate = np.asarray(masks[candidate_name], dtype=bool)
    control = np.asarray(masks[control_name], dtype=bool)
    union = candidate | control

    cheek_name = 'left_cheek' if side == 'screen_left' else 'right_cheek'
    if cheek_name not in masks:
        raise KeyError(f'必要な頬ROIがありません: {cheek_name}')
    cheek = np.asarray(masks[cheek_name], dtype=bool)

    ys, xs = np.where(union)
    if len(xs) == 0:
        raise ValueError(f'{side}: nasolabial union is empty')

    x0, y0, x1, y1 = xs.min(), ys.min(), xs.max(), ys.max()
    w = x1 - x0 + 1
    h = y1 - y0 + 1
    roi_w = max(18, round(0.95 * w))
    roi_h = max(18, round(1.10 * h))
    gap = max(4, round(0.10 * w))
    top = max(0, int(round(y0 - 0.15 * h)))
    bottom = min(image.shape[0], top + roi_h)

    if side == 'screen_left':
        right = max(0, int(x0 - gap))
        left = max(0, right - roi_w)
    elif side == 'screen_right':
        left = min(image.shape[1], int(x1 + gap))
        right = min(image.shape[1], left + roi_w)
    else:
        raise ValueError(f'unknown side: {side}')

    rect = np.zeros(image.shape[:2], dtype=bool)
    rect[top:bottom, left:right] = True
    mask = rect & cheek & ~union
    pixels = int(mask.sum())
    if pixels < 80:
        raise ValueError(f'{side}: upper cheek texture ROI is too small ({pixels} px)')
    return mask, (left, top, right, bottom)


def measure_upper_cheek_highpass(image_bgr, mask):
    image = np.asarray(image_bgr)
    if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:
        raise ValueError('Expected uint8 BGR image')
    lab = cv2.cvtColor(image.astype(np.float32) / 255.0, cv2.COLOR_BGR2LAB)
    lightness = lab[:, :, 0].astype(np.float64)
    low_frequency = cv2.GaussianBlur(lightness, (0, 0), TEXTURE_BLUR_SIGMA)
    highpass_abs = np.abs(lightness - low_frequency)
    base_l = float(np.median(lightness[mask]))
    if not np.isfinite(base_l) or base_l <= 1e-6:
        raise ValueError('upper cheek median L* is invalid')
    values = highpass_abs[mask]
    return {
        'highpass_median_pct': float(100.0 * np.percentile(values, 50) / base_l),
        'highpass_p90_pct': float(100.0 * np.percentile(values, 90) / base_l),
    }


def measure_upper_cheek_gvr(image_bgr, mask):
    image = np.asarray(image_bgr)
    if image.ndim != 3 or image.shape[2] != 3 or image.dtype != np.uint8:
        raise ValueError('Expected uint8 BGR image')
    intensity = np.rint(image.astype(np.float32).mean(axis=2)).astype(np.uint8)
    clahe = cv2.createCLAHE(
        clipLimit=GVR_INSPIRED_CLAHE_CLIP_LIMIT,
        tileGridSize=GVR_INSPIRED_CLAHE_TILE_GRID,
    )
    enhanced = clahe.apply(intensity)
    reflectance = np.clip(
        intensity.astype(np.int16) - enhanced.astype(np.int16),
        0,
        None,
    ).astype(np.float64)
    whole_sum = float(reflectance.sum())
    if not np.isfinite(whole_sum) or whole_sum <= 0:
        raise ValueError('GVR-inspired reflectance sum is invalid')
    return float(np.mean(reflectance[mask]) / whole_sum)


upper_cheek_rows = []
upper_cheek_review = []
for pair in rank_summary['pairs']:
    rank = pair['rank']
    selection = pair['selection']
    before_img, before_masks, _ = _load_phase(selection['before'], 'eye_texture')
    after_img, after_masks, _ = _load_phase(selection['after'], 'eye_texture')

    for side, side_label in (('screen_left', '画面左'), ('screen_right', '画面右')):
        before_roi, before_box = build_upper_cheek_texture_mask(before_img, before_masks, side)
        after_roi, after_box = build_upper_cheek_texture_mask(after_img, after_masks, side)
        before_hp = measure_upper_cheek_highpass(before_img, before_roi)
        after_hp = measure_upper_cheek_highpass(after_img, after_roi)
        before_gvr = measure_upper_cheek_gvr(before_img, before_roi)
        after_gvr = measure_upper_cheek_gvr(after_img, after_roi)

        metrics = (
            ('highpass_median_pct', before_hp['highpass_median_pct'], after_hp['highpass_median_pct']),
            ('highpass_p90_pct', before_hp['highpass_p90_pct'], after_hp['highpass_p90_pct']),
            ('gvr_inspired_ratio', before_gvr, after_gvr),
        )
        for metric, before_value, after_value in metrics:
            upper_cheek_rows.append({
                'rank': rank,
                'side': side_label,
                'metric': metric,
                'before': before_value,
                'after': after_value,
                'delta_after_minus_before': after_value - before_value,
            })

        upper_cheek_review.append({
            'rank': rank,
            'side': side,
            'side_label': side_label,
            'before_t': selection['before']['timestamp_seconds'],
            'after_t': selection['after']['timestamp_seconds'],
            'before_img': before_img,
            'after_img': after_img,
            'before_roi': before_roi,
            'after_roi': after_roi,
            'before_box': before_box,
            'after_box': after_box,
        })

upper_cheek_df = pd.DataFrame(upper_cheek_rows).sort_values(
    ['rank', 'side', 'metric']
).reset_index(drop=True)
expected_rows = len(rank_summary['pairs']) * 2 * 3
if len(upper_cheek_df) != expected_rows:
    raise RuntimeError(f'upper cheek rows mismatch: expected={expected_rows}, actual={len(upper_cheek_df)}')

summary_rows = []
for (metric, side), group in upper_cheek_df.groupby(['metric', 'side'], sort=True):
    values = group['delta_after_minus_before']
    positive = int((values > 0).sum())
    negative = int((values < 0).sum())
    zero = int((values == 0).sum())
    majority_count = max(positive, negative)
    summary_rows.append({
        'metric': metric,
        'side': side,
        'n': int(len(values)),
        'positive': positive,
        'negative': negative,
        'zero': zero,
        'consistency': majority_count / len(values),
        'median_delta': float(values.median()),
        'mean_delta': float(values.mean()),
    })
upper_cheek_summary = pd.DataFrame(summary_rows)
display(upper_cheek_summary)


In [ ]:
import base64
import html
from IPython.display import HTML, display

if 'upper_cheek_df' not in globals() or 'upper_cheek_review' not in globals():
    raise RuntimeError('upper cheek の集計結果がありません。直前のセルを実行してください。')

METRIC_LABELS = {
    'highpass_median_pct': '細かな質感コントラスト（中央値）',
    'highpass_p90_pct': '細かな質感コントラスト（p90）',
    'gvr_inspired_ratio': 'GVR-inspired 水分反射比',
}


def upper_cheek_crop_uri(image_bgr, mask, box):
    x0, y0, x1, y1 = box
    roi_w = max(1, x1 - x0)
    roi_h = max(1, y1 - y0)
    pad = max(18, round(max(roi_w, roi_h) * 0.35))
    left = max(0, x0 - pad)
    top = max(0, y0 - pad)
    right = min(image_bgr.shape[1], x1 + pad)
    bottom = min(image_bgr.shape[0], y1 + pad)
    if left >= right or top >= bottom:
        raise ValueError(f'invalid upper cheek crop: {(left, top, right, bottom)}')

    crop = image_bgr[top:bottom, left:right].copy()
    local_mask = np.asarray(mask[top:bottom, left:right], dtype=bool)
    if not np.any(local_mask):
        raise ValueError('upper cheek local mask is empty')
    fill = crop.copy()
    fill[local_mask] = (0, 175, 255)
    overlay = cv2.addWeighted(fill, 0.26, crop, 0.74, 0)
    contours = cv2.findContours(
        local_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )[0]
    cv2.drawContours(overlay, contours, -1, (0, 140, 255), 2, cv2.LINE_AA)
    ok, encoded = cv2.imencode('.png', overlay)
    if not ok:
        raise RuntimeError('failed to encode upper cheek crop')
    return 'data:image/png;base64,' + base64.b64encode(encoded.tobytes()).decode('ascii')


def format_metric_value(metric, value, signed=False):
    value = float(value)
    if metric == 'gvr_inspired_ratio':
        return f'{value:+.3e}' if signed else f'{value:.3e}'
    return f'{value:+.3f}' if signed else f'{value:.3f}'


review_lookup = {(item['rank'], item['side_label']): item for item in upper_cheek_review}
parts = [
    "<div class='upper-cheek-report'>",
    "<h1>上頬プレーン肌 ROI：20候補の詳細確認</h1>",
    "<p>オレンジ色が実際の計測ROIです。各ROIのアップの直下に、そのROIから計算した3指標を表示します。</p>",
]

for rank in sorted(upper_cheek_df['rank'].unique()):
    first = review_lookup[(rank, '画面左')]
    parts.append(
        f"<section><h2>rank {rank} &nbsp; "
        f"before {first['before_t']:.2f}s → after {first['after_t']:.2f}s</h2>"
    )
    for side_label in ('画面左', '画面右'):
        item = review_lookup[(rank, side_label)]
        before_uri = upper_cheek_crop_uri(item['before_img'], item['before_roi'], item['before_box'])
        after_uri = upper_cheek_crop_uri(item['after_img'], item['after_roi'], item['after_box'])
        metrics = upper_cheek_df[
            (upper_cheek_df['rank'] == rank) & (upper_cheek_df['side'] == side_label)
        ].set_index('metric')
        expected = {'highpass_median_pct', 'highpass_p90_pct', 'gvr_inspired_ratio'}
        if set(metrics.index) != expected:
            raise RuntimeError(f'rank {rank} {side_label}: metrics mismatch: {set(metrics.index)}')

        rows = []
        for metric in ('highpass_median_pct', 'highpass_p90_pct', 'gvr_inspired_ratio'):
            row = metrics.loc[metric]
            rows.append(
                '<tr>'
                f"<td>{html.escape(METRIC_LABELS[metric])}</td>"
                f"<td>{format_metric_value(metric, row['before'])}</td>"
                f"<td>{format_metric_value(metric, row['after'])}</td>"
                f"<td>{format_metric_value(metric, row['delta_after_minus_before'], signed=True)}</td>"
                '</tr>'
            )

        parts.append(
            f"<div class='side-card'><h3>{side_label}</h3>"
            "<div class='roi-pair'>"
            f"<figure><figcaption>before</figcaption><img src='{before_uri}'></figure>"
            f"<figure><figcaption>after</figcaption><img src='{after_uri}'></figure>"
            "</div>"
            "<table><thead><tr><th>指標</th><th>before</th><th>after</th><th>after - before</th></tr></thead>"
            f"<tbody>{''.join(rows)}</tbody></table></div>"
        )
    parts.append('</section>')
parts.append('</div>')

style = """
<style>
.upper-cheek-report{background:#fff!important;color:#17221d!important;color-scheme:light!important;padding:20px;font:15px/1.6 system-ui,sans-serif}
.upper-cheek-report section{background:#fff!important;border:1px solid #cfd8d3;border-radius:12px;padding:18px;margin:24px 0}
.upper-cheek-report .side-card{background:#fafcfb!important;border:1px solid #dce4df;border-radius:10px;padding:14px;margin:16px 0}
.upper-cheek-report h1,.upper-cheek-report h2,.upper-cheek-report h3,.upper-cheek-report p,.upper-cheek-report th,.upper-cheek-report td,.upper-cheek-report figcaption{color:#17221d!important}
.upper-cheek-report .roi-pair{display:grid;grid-template-columns:1fr 1fr;gap:14px;align-items:start}
.upper-cheek-report figure{margin:0;background:#fff!important}
.upper-cheek-report figcaption{font-weight:700;margin:0 0 6px}
.upper-cheek-report img{display:block;width:100%;max-height:420px;object-fit:contain;background:#fff!important;border:1px solid #dfe6e2;border-radius:8px}
.upper-cheek-report table{width:100%;border-collapse:collapse;margin-top:12px;background:#fff!important}
.upper-cheek-report th{background:#eef3f0!important;font-weight:700}
.upper-cheek-report th,.upper-cheek-report td{padding:8px 10px;border-bottom:1px solid #d7ded9;text-align:right}
.upper-cheek-report th:first-child,.upper-cheek-report td:first-child{text-align:left}
@media(max-width:800px){.upper-cheek-report .roi-pair{grid-template-columns:1fr}}
</style>
"""
detail_html = style + ''.join(parts)
detail_path = rank_run_dir / 'upper_cheek_detail_report.html'
detail_path.write_text(detail_html, encoding='utf-8')
display(HTML(detail_html))
print('upper cheek detail report:', detail_path)


## 眉下の皮膚と対照 ROI を拡大して確認する

眉下の皮膚に加えて、同じ固定フレームの左右頬・額を対照 ROI として before / after で拡大表示します。黄色が実際の計測領域です。


In [ ]:
from pathlib import Path
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

# notebook を repo 直下 / notebooks/ のどちらから実行しても同じ場所を指す。
cwd = Path.cwd().resolve()
marker = Path('analysis/analyze_selected_regions.py')
if (cwd / marker).is_file():
    project_root = cwd
elif cwd.name == 'notebooks' and (cwd.parent / marker).is_file():
    project_root = cwd.parent
    os.chdir(project_root)
else:
    raise RuntimeError(f'ikiikimake のリポジトリ直下または notebooks/ から実行してください: current={cwd}')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.analyze_selected_regions import ANALYSIS_VERSION, analyze_selected_regions, _load_phase
EXPECTED_ANALYSIS_VERSION = 'selected-region-appearance-v10'
if ANALYSIS_VERSION != EXPECTED_ANALYSIS_VERSION:
    raise RuntimeError(
        f'解析モジュールが古いです: loaded={ANALYSIS_VERSION}, expected={EXPECTED_ANALYSIS_VERSION}. '
        'Jupyter kernel を再起動して Run All してください。'
    )

video = Path('makeup0923.mp4')
output = Path('outputs') / video.stem
selected_path = output / 'selected_region_pairs.json'
analysis_root = output / 'selected_region_analysis'
if not selected_path.is_file():
    raise FileNotFoundError(selected_path)

closeup_summary = analyze_selected_regions(selected_path, analysis_root)
region = 'eye_texture'
if region not in closeup_summary['regions']:
    raise ValueError(f'{region} が selected_region_pairs.json にありません。')
selection = closeup_summary['regions'][region]['selection']
before_img, before_masks, _ = _load_phase(selection['before'], region)
after_img, after_masks, _ = _load_phase(selection['after'], region)

def make_closeup(image, mask, pad_face_fraction=0.035):
    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('ROI mask is empty')
    face_width = xs.max() - xs.min() + 1
    pad = max(12, round(face_width * pad_face_fraction))
    y0 = max(0, int(ys.min()) - pad)
    y1 = min(image.shape[0], int(ys.max()) + pad + 1)
    x0 = max(0, int(xs.min()) - pad)
    x1 = min(image.shape[1], int(xs.max()) + pad + 1)
    crop = image[y0:y1, x0:x1].copy()
    local_mask = mask[y0:y1, x0:x1]
    fill = crop.copy()
    fill[local_mask] = (50, 220, 240)
    overlay = cv2.addWeighted(fill, 0.35, crop, 0.65, 0)
    contours = cv2.findContours(local_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    cv2.drawContours(overlay, contours, -1, (0, 200, 255), 1, cv2.LINE_AA)
    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

mask_names = [
    ('screen_left_upper_lid_skin', '画面左眉下の皮膚'),
    ('screen_right_upper_lid_skin', '画面右眉下の皮膚'),
    ('left_cheek', '画面左頬・対照'),
    ('right_cheek', '画面右頬・対照'),
    ('forehead', '額・対照'),
]

fig, axes = plt.subplots(5, 2, figsize=(12, 15), constrained_layout=True)
for row, (mask_name, label) in enumerate(mask_names):
    axes[row, 0].imshow(make_closeup(before_img, before_masks[mask_name]))
    axes[row, 0].set_title(f'before / {label}')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(make_closeup(after_img, after_masks[mask_name]))
    axes[row, 1].set_title(f'after / {label}')
    axes[row, 1].axis('off')

run_dir = Path(closeup_summary['output_dir'])
closeup_path = run_dir / 'eye_texture_closeup.png'
fig.savefig(closeup_path, dpi=180, bbox_inches='tight')
plt.show()
print('closeup:', closeup_path)


## ほうれい線候補 ROI と周囲皮膚対照帯を目視確認する

小鼻から口角手前へ向かう二次Bezier曲線を頬側へふくらませた候補帯です。黄色がほうれい線候補、シアンがさらに頬側の周囲皮膚対照帯です。before / after は各側で同じクロップサイズにそろえて表示します。ここではまだシワそのものを検出・数値化しません。


In [ ]:
nasolabial_sides = [
    ('screen_left', '画面左'),
    ('screen_right', '画面右'),
]

def _mask_bounds(mask):
    mask = np.asarray(mask, dtype=bool)
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError('ROI mask is empty')
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())

def _fixed_crop_bounds(mask_union, crop_width, crop_height, image_shape):
    x0, y0, x1, y1 = _mask_bounds(mask_union)
    cx = 0.5 * (x0 + x1)
    cy = 0.5 * (y0 + y1)
    height, width = image_shape[:2]
    if crop_width > width or crop_height > height:
        raise ValueError('Requested fixed crop is larger than source image')
    left = int(round(cx - crop_width / 2))
    top = int(round(cy - crop_height / 2))
    left = min(max(0, left), width - crop_width)
    top = min(max(0, top), height - crop_height)
    return left, top, left + crop_width, top + crop_height

def _overlay_nasolabial(image, candidate, control, crop_width, crop_height):
    union = np.asarray(candidate, dtype=bool) | np.asarray(control, dtype=bool)
    x0, y0, x1, y1 = _fixed_crop_bounds(union, crop_width, crop_height, image.shape)
    crop = image[y0:y1, x0:x1].copy()
    candidate_local = np.asarray(candidate[y0:y1, x0:x1], dtype=bool)
    control_local = np.asarray(control[y0:y1, x0:x1], dtype=bool)

    fill = crop.copy()
    fill[candidate_local] = (50, 220, 240)
    fill[control_local] = (240, 220, 50)
    overlay = cv2.addWeighted(fill, 0.32, crop, 0.68, 0)

    candidate_contours = cv2.findContours(candidate_local.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    control_contours = cv2.findContours(control_local.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0]
    cv2.drawContours(overlay, candidate_contours, -1, (0, 200, 255), 1, cv2.LINE_AA)
    cv2.drawContours(overlay, control_contours, -1, (255, 200, 0), 1, cv2.LINE_AA)
    return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for row, (side, label) in enumerate(nasolabial_sides):
    candidate_name = f'{side}_nasolabial_candidate'
    control_name = f'{side}_nasolabial_outer_control'
    for name in (candidate_name, control_name):
        if name not in before_masks or name not in after_masks:
            raise RuntimeError(f'ほうれい線レビュー ROI がありません: {name}')

    before_union = before_masks[candidate_name] | before_masks[control_name]
    after_union = after_masks[candidate_name] | after_masks[control_name]
    before_box = _mask_bounds(before_union)
    after_box = _mask_bounds(after_union)
    roi_width = max(before_box[2] - before_box[0] + 1, after_box[2] - after_box[0] + 1)
    roi_height = max(before_box[3] - before_box[1] + 1, after_box[3] - after_box[1] + 1)
    pad = max(24, round(max(roi_width, roi_height) * 0.35))
    crop_width = roi_width + 2 * pad
    crop_height = roi_height + 2 * pad

    axes[row, 0].imshow(_overlay_nasolabial(
        before_img, before_masks[candidate_name], before_masks[control_name], crop_width, crop_height
    ))
    axes[row, 0].set_title(f'before / {label}')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(_overlay_nasolabial(
        after_img, after_masks[candidate_name], after_masks[control_name], crop_width, crop_height
    ))
    axes[row, 1].set_title(f'after / {label}')
    axes[row, 1].axis('off')

nasolabial_path = run_dir / 'nasolabial_curve_review.png'
fig.savefig(nasolabial_path, dpi=180, bbox_inches='tight')
plt.show()
print('nasolabial curve review:', nasolabial_path)
